# 01 Pdf Parsing

This notebook is part of the **Production-Style PDF Chatbot / RAG System Project**.

## Objectives
- Understand the underlying concepts.
- Build a production-quality implementation.
- Learn the theory behind every code block.


In [2]:
# ==========================================================
# Notebook 01: PDF Parsing
# ==========================================================

import os
import pdfplumber
import pandas as pd

In [3]:
PDF_PATH = "data/raw/annual_report.pdf"

In [4]:
with pdfplumber.open(PDF_PATH) as pdf:

    print("Total Pages:", len(pdf.pages))

Total Pages: 125


In [6]:
with pdfplumber.open(PDF_PATH) as pdf:

    first_page = pdf.pages[2]

    text = first_page.extract_text()

print(text[:1000])

CCoommppaannyy OOvveerrvviieeww SSttaattuuttoorryy RReeppoorrttss FFiinnaanncciiaall SSttaatteemmeennttss NNoottiiccee
AAbbbbootttt IInnddiiaa LLiimmiitteedd 22--5599 6600--115555 115566--222277 222288--224433
TOGETHER TOWARDS
WELLNESS
At Abbott India Limited, our commitment to wellness is
a shared journey. Rooted in our purpose to help people
live healthier, fuller lives, we bring trusted therapies and
innovative solutions that make a meaningful difference
across every stage of life.
Our approach spans the entire healthcare continuum
- from prevention and early intervention to treatment
and long-term care - ensuring that individuals receive
the support they need, wherever they are on their health
journey. With a broad and reliable portfolio across key
therapeutic areas such as Gastroenterology, Women’s
Health, Metabolics, Central Nervous System (CNS),
Vaccines and Multi-Specialty, we are addressing real-world
health challenges with compassion, science, and excellence.
Our strength lie

In [7]:
pages = []

with pdfplumber.open(PDF_PATH) as pdf:

    for page_number, page in enumerate(pdf.pages, start=1):

        text = page.extract_text()

        pages.append({"page_number": page_number, "page_text": text})

In [8]:
pages_df = pd.DataFrame(pages)

pages_df.head()

,page_number,page_text
0,1,ABBOTT INDIA LIMITED\nANNUAL REPORT 2024-25\nT...
1,2,Abbott India Limited\nCONTENTS\nCompany Overvi...
2,3,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...
3,4,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...
4,5,Company Overview Statutory Reports Financial S...


In [9]:
pages_df["page_text"].isnull().sum()

0

In [10]:
pages_df = pages_df.dropna(subset=["page_text"])

In [11]:
def clean_text(text):

    if text is None:
        return ""

    text = text.replace("\n", " ")

    text = " ".join(text.split())

    return text.strip()

In [12]:
pages_df["clean_text"] = pages_df["page_text"].apply(clean_text)

pages_df.head()

,page_number,page_text,clean_text
0,1,ABBOTT INDIA LIMITED\nANNUAL REPORT 2024-25\nT...,ABBOTT INDIA LIMITED ANNUAL REPORT 2024-25 TOG...
1,2,Abbott India Limited\nCONTENTS\nCompany Overvi...,Abbott India Limited CONTENTS Company Overview...
2,3,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...
3,4,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...
4,5,Company Overview Statutory Reports Financial S...,Company Overview Statutory Reports Financial S...


In [13]:
document_name = os.path.basename(PDF_PATH)

pages_df["source_document"] = document_name

In [14]:
pages_df["source_document"]

0      annual_report.pdf
1      annual_report.pdf
2      annual_report.pdf
3      annual_report.pdf
4      annual_report.pdf
             ...        
120    annual_report.pdf
121    annual_report.pdf
122    annual_report.pdf
123    annual_report.pdf
124    annual_report.pdf
Name: source_document, Length: 125, dtype: object

In [15]:
OUTPUT_PATH = "data/processed/" "parsed_documents.csv"

pages_df.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)

Saved to: data/processed/parsed_documents.csv


In [16]:
def parse_pdf(pdf_path):

    pages = []

    with pdfplumber.open(pdf_path) as pdf:

        for page_number, page in enumerate(pdf.pages, start=1):

            text = page.extract_text()

            pages.append(
                {
                    "page_number": page_number,
                    "page_text": clean_text(text),
                    "source_document": os.path.basename(pdf_path),
                }
            )

    return pd.DataFrame(pages)

In [17]:
parsed_df = parse_pdf("data/raw/annual_report.pdf")

parsed_df.head()

,page_number,page_text,source_document
0,1,ABBOTT INDIA LIMITED ANNUAL REPORT 2024-25 TOG...,annual_report.pdf
1,2,Abbott India Limited CONTENTS Company Overview...,annual_report.pdf
2,3,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...,annual_report.pdf
3,4,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...,annual_report.pdf
4,5,Company Overview Statutory Reports Financial S...,annual_report.pdf


In [18]:
all_documents = []

RAW_FOLDER = "data/raw"

for file in os.listdir(RAW_FOLDER):

    if file.endswith(".pdf"):

        pdf_path = os.path.join(RAW_FOLDER, file)

        df = parse_pdf(pdf_path)

        all_documents.append(df)

In [19]:
corpus_df = pd.concat(all_documents, ignore_index=True)

corpus_df.head()

,page_number,page_text,source_document
0,1,ABBOTT INDIA LIMITED ANNUAL REPORT 2024-25 TOG...,annual_report.pdf
1,2,Abbott India Limited CONTENTS Company Overview...,annual_report.pdf
2,3,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...,annual_report.pdf
3,4,CCoommppaannyy OOvveerrvviieeww SSttaattuuttoo...,annual_report.pdf
4,5,Company Overview Statutory Reports Financial S...,annual_report.pdf
